In [ ]:
# CareerLens — run this cell first (inline plots).

%matplotlib inline


In [ ]:
# =============================================================================
# CareerLens — shared engine (run Cell 0 first: %%matplotlib inline)
# =============================================================================
# Use run_pipeline() below: one call per domain (like the original CSE example),
# or run_pipeline_auto() to pick the best-matching domain from your skills.
# =============================================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import random
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBRegressor

_TF = dict(token_pattern=r"(?u)\b\w+\b")


def _ensure_project_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents][:10]:
        if (base / "dataset" / "computer_science_ai.csv").is_file():
            os.chdir(base)
            return base
    raise FileNotFoundError(
        "Cannot find dataset/computer_science_ai.csv. Open the fulldevpro folder as your workspace "
        "(the folder that contains dataset/ and proj.ipynb), restart the kernel, run again."
    )


print("Project root:", _ensure_project_root())


def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z, ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def standardize_skills(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    mapping = {
        "ml": "machine learning",
        "dl": "deep learning",
        "nlp": "natural language processing",
        "ai": "artificial intelligence",
        "cv": "computer vision",
        "ds": "data science",
    }
    for k, v in mapping.items():
        text = re.sub(rf"\b{k}\b", v, text)
    return ",".join(
        [s.strip().replace(" ", "_") for s in text.split(",") if s.strip()]
    )


def clean_split(text):
    return {i.strip() for i in text.split(",") if i.strip() != ""}


CAREER_FIELD_FILES = {
    "Computer Science & AI": "dataset/computer_science_ai.csv",
    "Mechanical Engineering": "dataset/mechanical_engineering.csv",
    "Civil Engineering": "dataset/civil_engineering.csv",
    "Electrical Engineering": "dataset/electrical_engineering.csv",
    "Electronics & Communication Engineering": "dataset/electronics_communication_engineering.csv",
    "Textile": "dataset/textile.csv",
    "Medicine": "dataset/medicine.csv",
    "Finance": "dataset/finance.csv",
}

CAREER_DOMAINS_ORDERED = list(CAREER_FIELD_FILES.keys())

# When two domains get the same TF-IDF score, prefer specialist domains over CS (fixes "always CSE" ties)
_AUTO_TIEBREAK = {
    "Civil Engineering": 8,
    "Mechanical Engineering": 7,
    "Electrical Engineering": 6,
    "Electronics & Communication Engineering": 5,
    "Textile": 4,
    "Medicine": 3,
    "Finance": 2,
    "Computer Science & AI": 1,
}

CAREER_BASE_SALARY = {
    "Computer Science & AI": {
        "AI Engineer": 8,
        "ML Engineer": 6,
        "Data Scientist": 4,
        "Data Analyst": 3,
        "Frontend Developer": 3,
        "Backend Developer": 3,
        "Full Stack Developer": 6,
        "DevOps Engineer": 6,
    },
    "Mechanical Engineering": {
        "Design Engineer": 6,
        "Manufacturing Engineer": 5,
        "Quality Engineer": 5,
        "HVAC Engineer": 6,
        "Maintenance Engineer": 4,
        "Robotics Automation Engineer": 7,
        "Project Engineer": 6,
        "CAE Engineer": 7,
    },
    "Civil Engineering": {
        "Structural Engineer": 7,
        "Site Engineer": 4,
        "Quantity Surveyor": 5,
        "Geotechnical Engineer": 6,
        "Highway Engineer": 5,
        "BIM Civil Engineer": 6,
        "Project Manager Civil": 7,
        "Urban Planner": 5,
    },
    "Electrical Engineering": {
        "Power Systems Engineer": 7,
        "Control Systems Engineer": 6,
        "Electrical Design Engineer": 6,
        "Substation Engineer": 7,
        "Renewable Energy Engineer": 6,
        "Electrical Maintenance Engineer": 4,
        "Electrical Project Engineer": 6,
        "Protection Engineer": 7,
    },
    "Electronics & Communication Engineering": {
        "VLSI Design Engineer": 8,
        "Embedded Systems Engineer": 7,
        "RF Engineer": 7,
        "Analog Design Engineer": 8,
        "PCB Design Engineer": 6,
        "Hardware Test Engineer": 5,
        "DSP Engineer": 7,
        "Telecom Network Engineer": 6,
    },
    "Textile": {
        "Textile Technologist": 4,
        "Fabric Development Specialist": 5,
        "Quality Control Textile": 4,
        "Fashion Production Manager": 6,
        "Knitting Engineer": 5,
        "Dyeing Technologist": 5,
        "Merchandiser": 4,
        "Supply Chain Textile": 5,
    },
    "Medicine": {
        "General Physician": 8,
        "Surgeon": 9,
        "Pediatrician": 8,
        "Radiologist": 8,
        "Pathologist": 7,
        "Emergency Medicine Doctor": 8,
        "Psychiatrist": 7,
        "Cardiologist": 9,
    },
    "Finance": {
        "Financial Analyst": 5,
        "Investment Banker": 8,
        "Chartered Accountant": 7,
        "Risk Analyst": 6,
        "Portfolio Manager": 8,
        "Financial Controller": 7,
        "Tax Consultant": 6,
        "Credit Analyst": 5,
    },
}


def load_and_prepare_domain(domain):
    path = Path(CAREER_FIELD_FILES[domain])
    if not path.is_file():
        raise FileNotFoundError(f"Missing: {Path.cwd() / path}")
    d = pd.read_csv(path)
    d.rename(
        columns={
            "Job_Name": "job_title",
            "Required_Skills": "skills_required",
            "User_Learned_Skills": "user_skills",
            "Avg_Salary_LPA": "salary",
            "Job_Postings": "postings",
            "Year": "year",
        },
        inplace=True,
    )
    d["skills_required"] = d["skills_required"].apply(normalize_text).apply(standardize_skills)
    d["user_skills"] = d["user_skills"].apply(normalize_text).apply(standardize_skills)
    return d


def pick_best_domain_for_skills(skills_cleaned, verbose=True):
    """
    Pick domain with highest max cosine sim(user, job_row required skills).
    Tie-break: prefer Civil/ME/... over Computer Science & AI when scores are equal.
    """
    best_key = None
    best_domain = CAREER_DOMAINS_ORDERED[0]
    best_sim = -1.0
    n = len(CAREER_DOMAINS_ORDERED)

    if verbose:
        print(f"Auto: scoring skills across {n} domains (this can take ~1 minute)…", flush=True)

    for i, domain in enumerate(CAREER_DOMAINS_ORDERED, 1):
        if verbose:
            print(f"  [{i}/{n}] {domain} …", flush=True)
        d = load_and_prepare_domain(domain)
        tv = TfidfVectorizer(stop_words="english", **_TF)
        tv.fit(list(d["skills_required"]) + list(d["user_skills"]))
        req_m = tv.transform(d["skills_required"])
        uv = tv.transform([skills_cleaned])
        sims = cosine_similarity(uv, req_m).flatten()
        bs = float(sims.max())
        tie = _AUTO_TIEBREAK.get(domain, 0)
        key = (bs, tie)
        if best_key is None or key > best_key:
            best_key = key
            best_domain = domain
            best_sim = bs

    return best_domain, best_sim


def run_pipeline(locked_domain, skills_str, year, show_plots=True, verbose=True):
    """
    Same logic as the original CSE notebook, for ONE domain.

    locked_domain: e.g. "Civil Engineering" or "Computer Science & AI"
    skills_str: comma-separated skills
    year: target year for salary prediction
    """
    skills = standardize_skills(normalize_text(skills_str))
    career_field_resolved = locked_domain

    if verbose:
        print(f"\n>>> Locked domain: {career_field_resolved}")

    df = load_and_prepare_domain(career_field_resolved)
    if verbose:
        print(f"    Loaded {len(df):,} rows from {CAREER_FIELD_FILES[career_field_resolved]}")

    tfidf = TfidfVectorizer(stop_words="english", **_TF)
    tfidf.fit(list(df["skills_required"]) + list(df["user_skills"]))
    req_matrix = tfidf.transform(df["skills_required"])
    user_matrix = tfidf.transform(df["user_skills"])

    df["similarity"] = [
        cosine_similarity(req_matrix[i], user_matrix[i])[0][0]
        for i in range(len(df))
    ]

    base_salary = CAREER_BASE_SALARY[career_field_resolved]
    default_base = 8 if career_field_resolved == "Computer Science & AI" else 6
    df["salary"] = [
        base_salary.get(df["job_title"].iloc[i], default_base)
        + df["similarity"].iloc[i] * 15
        + random.uniform(-0.5, 0.5)
        for i in range(len(df))
    ]

    df["num_user_skills"] = df["user_skills"].apply(lambda x: len(clean_split(x)))
    df["num_required_skills"] = df["skills_required"].apply(lambda x: len(clean_split(x)))
    df["skill_match_percent"] = df["similarity"] * 100

    le = LabelEncoder()
    df["job_encoded"] = le.fit_transform(df["job_title"])

    if show_plots:
        plt.figure(figsize=(8, 5))
        n, bins, patches = plt.hist(df["salary"], bins=30, edgecolor="white")
        for patch, color in zip(patches, plt.cm.plasma(np.linspace(0.2, 0.9, len(patches)))):
            patch.set_facecolor(color)
        plt.title(f"Salary Distribution — {career_field_resolved}")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 5))
        sc = plt.scatter(df["similarity"], df["salary"], c=df["salary"], cmap="coolwarm", alpha=0.6)
        plt.colorbar(sc)
        plt.title("Skill Match vs Salary")
        plt.tight_layout()
        plt.show()

        avg_salary = df.groupby("job_title")["salary"].mean().sort_values(ascending=False)
        plt.figure(figsize=(9, 5))
        plt.bar(avg_salary.index, avg_salary.values, color=plt.cm.Set2(np.linspace(0, 1, len(avg_salary))))
        plt.xticks(rotation=30)
        plt.title(f"Avg Salary by Role — {career_field_resolved}")
        plt.tight_layout()
        plt.show()

    X = df[
        [
            "year",
            "postings",
            "job_encoded",
            "similarity",
            "num_user_skills",
            "num_required_skills",
            "skill_match_percent",
        ]
    ]
    y = np.log1p(df["salary"])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(n_estimators=500, max_depth=8, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred_actual = np.expm1(y_pred)
    y_test_actual = np.expm1(y_test)
    tolerance = 0.1
    correct = np.abs(y_pred_actual - y_test_actual) <= tolerance * y_test_actual
    accuracy = np.mean(correct) * 100

    if verbose:
        print("\n========== MODEL PERFORMANCE ==========")
        print(f"Tolerance Accuracy (+/-10%) : {round(accuracy, 2)}%")
        print(f"R2 Score                    : {round(r2_score(y_test, y_pred), 4)}")
        print(f"MAE                         : {round(mean_absolute_error(y_test_actual, y_pred_actual), 2)}")
        print("=======================================")

    user_vec = tfidf.transform([skills])
    sims = cosine_similarity(user_vec, req_matrix).flatten()
    best_idx = int(np.argmax(sims))
    best_job = df["job_title"].iloc[best_idx]
    best_sim = float(sims[best_idx])

    job_encoded = le.transform([best_job])[0]
    input_df = pd.DataFrame(
        [
            [
                year,
                df["postings"].mean(),
                job_encoded,
                best_sim,
                len(clean_split(skills)),
                len(clean_split(df["skills_required"].iloc[best_idx])),
                best_sim * 100,
            ]
        ],
        columns=X.columns,
    )
    salary = float(np.expm1(model.predict(input_df))[0])

    user_set = clean_split(skills)
    req_set = set()
    for row in df[df["job_title"] == best_job]["skills_required"]:
        req_set.update(clean_split(row))
    missing = req_set - user_set
    matched = user_set & req_set

    if verbose:
        print("\n==============================")
        print(f"Career domain used    : {career_field_resolved}")
        print(f"Best Job Match        : {best_job}")
        print(f"Match Score           : {round(best_sim * 100, 2)}%")
        print(f"Predicted Salary      : {round(salary)} LPA")
        print("\nYOUR SKILLS:")
        for s in sorted(user_set):
            print(" +", s)
        print("\nREQUIRED SKILLS (role):")
        for s in sorted(req_set):
            print(" *", s)
        print("\nMISSING SKILLS:")
        if missing:
            for s in sorted(missing):
                print(" -", s)
        else:
            print(" (none)")
        print("\nMATCHED SKILLS:")
        for s in sorted(matched):
            print(" =", s)
        print("==============================")

    joblib.dump(
        {"best_job": best_job, "df": df, "career_field": career_field_resolved},
        "salary_output.pkl",
    )

    return {
        "domain": career_field_resolved,
        "best_job": best_job,
        "best_sim": best_sim,
        "salary": salary,
        "skills": skills,
    }


def run_pipeline_auto(skills_str, year, show_plots=True, verbose=True):
    """Pick the strongest domain for these skills, then run the same pipeline as run_pipeline."""
    skills = standardize_skills(normalize_text(skills_str))
    dom, preview = pick_best_domain_for_skills(skills, verbose=verbose)
    if verbose:
        print(f"\n>>> Auto-selected domain: {dom} (preview max similarity: {round(preview * 100, 2)}%)")
    if preview < 0.2 and verbose:
        print("Warning: weak match in every domain — try more role-specific keywords.")
    return run_pipeline(dom, skills_str, year, show_plots=show_plots, verbose=verbose)


print("Ready. Examples:")
print('  run_pipeline("Civil Engineering", "revit, staad pro, estimation", 2025)')
print('  run_pipeline_auto("python, pytorch, sql", 2025)')


## Same logic as the original CSE example — one cell per domain

Each domain cell calls `run_pipeline("<Domain>", "<skills>", year)` and loads **only** that domain CSV from `dataset/`. Civil skills match **civil** job titles only; CS is separate.

- Edit the skill string in any cell to experiment.
- **Auto cell** uses `run_pipeline_auto(...)`: picks the best-matching domain. If scores tie, **Civil / Mechanical / … win over Computer Science** so results are not biased to CSE.

**Order:** Cell 0 → Cell 1 (engine) → pick **one** domain cell (or Auto) → then **Job posting forecast** (needs `salary_output.pkl`).


In [ ]:
# Civil Engineering — edit skills if you want
run_pipeline("Civil Engineering", "revit, staad pro, estimation, gis, site supervision", 2025)


In [ ]:
# Mechanical Engineering — edit skills if you want
run_pipeline("Mechanical Engineering", "solidworks, cad, fea, plc, lean six sigma", 2025)


In [ ]:
# Electrical Engineering — edit skills if you want
run_pipeline("Electrical Engineering", "plc programming, scada, power systems, relay coordination, autocad electrical", 2025)


In [ ]:
# Electronics & Communication Engineering — edit skills if you want
run_pipeline("Electronics & Communication Engineering", "verilog, rtl design, embedded systems, rf engineer, altium designer", 2025)


In [ ]:
# Textile — edit skills if you want
run_pipeline("Textile", "weaving, dyeing chemistry, fabric testing, merchandiser, knitting", 2025)


In [ ]:
# Medicine — edit skills if you want
run_pipeline("Medicine", "clinical examination, diagnosis, patient care, emr, triage", 2025)


In [ ]:
# Finance — edit skills if you want
run_pipeline("Finance", "financial modeling, excel, valuation, gst, risk analysis", 2025)


In [ ]:
# Computer Science & AI — edit skills if you want
run_pipeline("Computer Science & AI", "python, machine learning, pytorch, sql, docker", 2025)


In [ ]:
# Auto: picks strongest domain for these skills (ties favor specialist over CS)
run_pipeline_auto("revit, staad pro, primavera, boq", 2025)


In [ ]:
# Job posting forecast (run after any run_pipeline above)

# ==============================
# 1 Import Libraries
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor

# ==============================
# 2 Load best_job from salary model
# ==============================
data     = joblib.load('salary_output.pkl')
best_job = data['best_job']
df       = data['df']
career_field = data.get('career_field', 'Computer Science & AI')

print(f"OK Loaded best job: {best_job} (domain: {career_field})")

# ==============================
# 3 Aggregate postings per year
# ==============================
job_yearly = df[df['job_title'] == best_job].groupby('year')['postings'].mean().reset_index()
job_yearly = job_yearly.sort_values('year').reset_index(drop=True)
job_yearly['postings'] = job_yearly['postings'].round().astype(int)

print(f"\nYearly Avg Postings for '{best_job}':")
print(job_yearly.to_string(index=False))

# ==============================
# 3.1 ROLE-BASED REALISTIC TREND
# ==============================
np.random.seed(42)

base = job_yearly['postings'].values
job_lower = best_job.lower()

if career_field == "Medicine":
    trend = np.linspace(0, 75, len(base))
elif career_field == "Finance":
    trend = np.linspace(0, 40, len(base))
elif career_field == "Civil Engineering":
    trend = np.linspace(0, 32, len(base))
elif career_field == "Mechanical Engineering":
    trend = np.linspace(0, 28, len(base))
elif career_field == "Electrical Engineering":
    trend = np.linspace(0, 30, len(base))
elif career_field == "Electronics & Communication Engineering":
    trend = np.linspace(0, 35, len(base))
elif career_field == "Textile":
    trend = np.linspace(0, -12, len(base))
elif "ai" in job_lower or "machine learning" in job_lower or "data scientist" in job_lower:
    trend = np.linspace(0, 80, len(base))
elif "backend" in job_lower or "frontend" in job_lower or "web" in job_lower:
    trend = np.linspace(0, 20, len(base))
elif "qa" in job_lower or "testing" in job_lower:
    trend = np.linspace(0, -30, len(base))
else:
    trend = np.linspace(0, 15, len(base))

noise = np.random.normal(0, 8, len(base))
job_yearly['postings'] = (base + trend + noise).round().astype(int)

print("\nOK Adjusted postings:")
print(job_yearly.to_string(index=False))

# ==============================
# 4 Feature Engineering
# ==============================
job_yearly['lag1'] = job_yearly['postings'].shift(1)
job_yearly['lag2'] = job_yearly['postings'].shift(2)
job_yearly['rolling_mean'] = job_yearly['postings'].rolling(2).mean()
job_yearly.dropna(inplace=True)
job_yearly.reset_index(drop=True, inplace=True)

job_yearly['trend_idx'] = range(len(job_yearly))
job_yearly['year_sq'] = job_yearly['year'] ** 2

X = job_yearly[['year', 'trend_idx', 'year_sq', 'lag1', 'lag2', 'rolling_mean']].values
y = job_yearly['postings'].values
years_arr = job_yearly['year'].values

# ==============================
# 5 Train Model
# ==============================
model = GradientBoostingRegressor(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8
)

model.fit(X, y)

# ==============================
# 6 Predictions
# ==============================
y_all_pred = model.predict(X)

r2  = r2_score(y, y_all_pred)
mae = mean_absolute_error(y, y_all_pred)

tolerance = 0.01
correct = np.abs(y - y_all_pred) <= tolerance * y
acc = (np.sum(correct) / len(y)) * 100

mape = np.mean(np.abs((y - y_all_pred) / y)) * 100

print("\n====== JOB POSTING FORECAST ACCURACY ======")
print(f"R2 Score : {round(r2, 4)}")
print(f"MAE      : {round(mae, 2)}")
print(f"MAPE     : {round(mape, 2)}%")
print(f"Accuracy : {round(acc, 2)}%")
print("============================================")

# ==============================
# 7 Forecast Next 5 Years (INDUSTRY LEVEL)
# ==============================

# Role-based growth rate (aligned with career domain)
if career_field == "Medicine":
    growth_rate = 0.048
elif career_field == "Finance":
    growth_rate = 0.035
elif career_field == "Civil Engineering":
    growth_rate = 0.032
elif career_field == "Mechanical Engineering":
    growth_rate = 0.028
elif career_field == "Electrical Engineering":
    growth_rate = 0.03
elif career_field == "Electronics & Communication Engineering":
    growth_rate = 0.033
elif career_field == "Textile":
    growth_rate = -0.008
elif "ai" in job_lower or "machine learning" in job_lower:
    growth_rate = 0.05
elif "backend" in job_lower or "frontend" in job_lower:
    growth_rate = 0.025
elif "qa" in job_lower or "testing" in job_lower:
    growth_rate = -0.01
else:
    growth_rate = 0.03

future_years = [int(years_arr[-1]) + i for i in range(1, 6)]
future_postings = []

last_vals = list(job_yearly['postings'].values)
max_trend = len(job_yearly) - 1

for i, yr in enumerate(future_years):

    lag1 = last_vals[-1]
    lag2 = last_vals[-2]
    rolling_mean = (lag1 + lag2) / 2

    trend_idx = max_trend + i + 1
    year_sq = yr ** 2

    row = np.array([[yr, trend_idx, year_sq, lag1, lag2, rolling_mean]])

    pred = model.predict(row)[0]

    growth_factor = 1 + (growth_rate * (i+1))

    noise = np.random.normal(0, 5)

    pred = pred * growth_factor + noise

    prev = last_vals[-1]
    pred = max(pred, prev * (1 + growth_rate/2))

    pred = round(pred)

    future_postings.append(pred)
    last_vals.append(pred)

print(f"\nJob Posting Forecast for '{best_job}' (Next 5 Years):")
for yr, val in zip(future_years, future_postings):
    print(f"   {yr} -> {val} avg postings")

# ==============================
# 8 Graph
# ==============================
plt.figure(figsize=(10, 5))

plt.plot(years_arr, y,
         label='Actual', color='royalblue',
         linewidth=2, marker='o')

plt.plot(years_arr, y_all_pred,
         label='Predicted', color='tomato',
         linewidth=2, linestyle='--', marker='s')

plt.plot(future_years, future_postings,
         label='Forecast (5Y)', color='green',
         linewidth=2, linestyle=':', marker='^')

for yr, val in zip(future_years, future_postings):
    plt.annotate(f'{val}', (yr, val),
                 textcoords="offset points", xytext=(0, 10),
                 ha='center', fontsize=9,
                 color='green', fontweight='bold')

plt.fill_between(years_arr, y, y_all_pred,
                 alpha=0.15, color='orange')

plt.title(f"Job Posting Forecast — {best_job}\n"
          f"Accuracy: {round(acc,2)}%",
          fontsize=13, fontweight='bold')

plt.xlabel("Year")
plt.ylabel("Avg Job Postings")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ==============================
# 9 Save
# ==============================
joblib.dump(model, "job_posting_model.pkl")
print("\nOK Model Saved!")
